# Step 1 : Install Dependencies

In [7]:
!pip install catboost geohash2 numpy scikit-learn lightgbm xgboost optuna -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


# Step 2 : Import Libraries

In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.metrics import r2_score

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
import geohash2
import warnings

warnings.filterwarnings('ignore')

# Step 3 : Load Dataset

In [9]:
train = pd.read_csv("dataset/train.csv")
test = pd.read_csv("dataset/test.csv")
sample = pd.read_csv("dataset/sample_submission.csv")

print(train.shape)
print(test.shape)

(77299, 11)
(41778, 10)


# Step 4 : Feature Engineering

## Convert Timestamp into Useful Features

In [10]:
def process_time(df):
    
    # Convert timestamp into string
    df['timestamp'] = df['timestamp'].astype(str)
    
    # Split hour and minute
    time_split = df['timestamp'].str.split(':', expand=True)
    
    df['hour'] = time_split[0].astype(int)
    df['minute'] = time_split[1].astype(int)
    
    # Total minutes in day
    df['total_minutes'] = df['hour'] * 60 + df['minute']
    
    # Day of week from numeric day column (day % 7 → 0=Mon...6=Sun)
    df['day_of_week'] = df['day'].astype(int) % 7
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    return df

train = process_time(train)
test = process_time(test)

## Decode Geohash Features

### Geohash contains spatial information.

In [11]:
def decode_geohash(df):

    latitudes = []
    longitudes = []
    
    for gh in df['geohash']:
    
        try:
            lat, lon = geohash2.decode_exactly(str(gh))[:2]
    
        except:
            lat = np.nan
            lon = np.nan
    
        latitudes.append(lat)
        longitudes.append(lon)
    
    df['latitude'] = latitudes
    df['longitude'] = longitudes
    
    return df

train = decode_geohash(train)
test = decode_geohash(test)

## Create Cyclic Time Features

### Traffic patterns are cyclic.

In [12]:
train['hour_sin'] = np.sin(2 * np.pi * train['hour'] / 24) 
train['hour_cos'] = np.cos(2 * np.pi * train['hour'] / 24) 

test['hour_sin'] = np.sin(2 * np.pi * test['hour'] / 24) 
test['hour_cos'] = np.cos(2 * np.pi * test['hour'] / 24)


train['minute_sin'] = np.sin(2 * np.pi * train['minute'] / 60) 
train['minute_cos'] = np.cos(2 * np.pi * train['minute'] / 60) 

test['minute_sin'] = np.sin(2 * np.pi * test['minute'] / 60) 
test['minute_cos'] = np.cos(2 * np.pi * test['minute'] / 60)


train['dow_sin'] = np.sin(2 * np.pi * train['day_of_week'] / 7) 
train['dow_cos'] = np.cos(2 * np.pi * train['day_of_week'] / 7) 

test['dow_sin'] = np.sin(2 * np.pi * test['day_of_week'] / 7) 
test['dow_cos'] = np.cos(2 * np.pi * test['day_of_week'] / 7)

# Interation Features & Target Encoding 

In [13]:
# --- Interaction Features ---
for df in [train, test]:
    df['lat_lon_interaction'] = df['latitude'] * df['longitude']
    df['lanes_x_hour'] = df['NumberofLanes'] * df['hour']
    df['temp_x_hour'] = df['Temperature'] * df['hour']

# --- Target Encoding for geohash (CV-safe) ---
# Use 5-fold target encoding on train to avoid leakage
from sklearn.model_selection import KFold as KF

train['geohash_demand_mean'] = np.nan
kf = KF(n_splits=5, shuffle=True, random_state=42)

for tr_idx, val_idx in kf.split(train):
    means = train.iloc[tr_idx].groupby('geohash')['demand'].mean()
    train.loc[train.index[val_idx], 'geohash_demand_mean'] = \
        train.iloc[val_idx]['geohash'].map(means)

# Global mean for any unmapped values
global_mean = train['demand'].mean()
train['geohash_demand_mean'] = train['geohash_demand_mean'].fillna(global_mean)

# For test: use full train mapping
geohash_means = train.groupby('geohash')['demand'].mean()
test['geohash_demand_mean'] = test['geohash'].map(geohash_means).fillna(global_mean)

# --- Target Encoding for day ---
train['day_demand_mean'] = np.nan
for tr_idx, val_idx in kf.split(train):
    means = train.iloc[tr_idx].groupby('day')['demand'].mean()
    train.loc[train.index[val_idx], 'day_demand_mean'] = \
        train.iloc[val_idx]['day'].map(means)
train['day_demand_mean'] = train['day_demand_mean'].fillna(global_mean)

day_means = train.groupby('day')['demand'].mean()
test['day_demand_mean'] = test['day'].map(day_means).fillna(global_mean)

# --- Log-transform target ---
train['demand_log'] = np.log1p(train['demand'])

print("New features added. Train shape:", train.shape)

New features added. Train shape: (77299, 30)


# Step 5 : Define Features

In [14]:
TARGET = 'demand_log'  # Use log-transformed target

features = [
    'geohash',
    'day',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    'hour',
    'minute',
    'total_minutes',
    'latitude',
    'longitude',
    'hour_sin',
    'hour_cos',
    'minute_sin',
    'minute_cos',
    'is_weekend',
    'day_of_week',
    'dow_sin',
    'dow_cos',
    'lat_lon_interaction',
    'lanes_x_hour',
    'temp_x_hour',
    'geohash_demand_mean',
    'day_demand_mean',
]

# Features for LightGBM/XGBoost (no raw categoricals — label encode them)
features_gbm = [f for f in features if f not in ['geohash', 'day', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']]

X = train[features].copy()
y = train[TARGET]

X_test = test[features].copy()

print(f"Features: {len(features)}, Train: {X.shape}, Test: {X_test.shape}")

Features: 26, Train: (77299, 26), Test: (41778, 26)


# Step 6 : Define Categorical Features

In [15]:
cat_features = [
    'geohash',
    'day',
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

# Convert categorical columns to string for CatBoost
for col in cat_features:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

# =========================
# HANDLE MISSING VALUES
# =========================
# Numerical columns — group-based imputation for Temperature
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    median_value = X[col].median()
    X[col] = X[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

# =========================
# LABEL ENCODE for LightGBM / XGBoost
# =========================
from sklearn.preprocessing import LabelEncoder

X_lgb = X.copy()
X_test_lgb = X_test.copy()

label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    le.fit(pd.concat([X_lgb[col], X_test_lgb[col]]))
    X_lgb[col] = le.transform(X_lgb[col])
    X_test_lgb[col] = le.transform(X_test_lgb[col])
    label_encoders[col] = le

print("Preprocessing done.")

Preprocessing done.


# Step 7 : Train Validation Split 

In [16]:
# Time-based split: train on earlier days, validate on last day
max_day = train['day'].max()
train_mask = train['day'] < max_day
valid_mask = train['day'] == max_day

X_train, X_valid = X[train_mask], X[valid_mask]
y_train, y_valid = y[train_mask], y[valid_mask]

X_train_lgb, X_valid_lgb = X_lgb[train_mask], X_lgb[valid_mask]

print(f"Train: {X_train.shape}, Valid: {X_valid.shape}")
print(f"Train days: < {max_day}, Valid day: {max_day}")

Train: (69427, 26), Valid: (7872, 26)
Train days: < 49, Valid day: 49


# Step 8 : Train CatBoost Model

In [17]:
# ========== CatBoost ==========
cat_model = CatBoostRegressor(
    iterations=7000,
    learning_rate=0.02,
    depth=8,
    l2_leaf_reg=5,
    bagging_temperature=0.8,
    random_strength=1.0,
    min_data_in_leaf=20,
    loss_function='RMSE',
    eval_metric='R2',
    random_seed=42,
    verbose=500,
    early_stopping_rounds=300
)

cat_model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

# ========== LightGBM ==========
lgb_model = LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.02,
    max_depth=8,
    num_leaves=127,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=5,
    random_state=42,
    verbose=-1
)

lgb_cat_indices = [features.index(c) for c in cat_features]
lgb_model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_valid_lgb, y_valid)],
    callbacks=[
        __import__('lightgbm').early_stopping(300),
        __import__('lightgbm').log_evaluation(500)
    ],
    categorical_feature=lgb_cat_indices
)

# ========== XGBoost ==========
xgb_model = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.02,
    max_depth=8,
    min_child_weight=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=5,
    random_state=42,
    verbosity=0,
    early_stopping_rounds=300,
    enable_categorical=True
)

xgb_model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_valid_lgb, y_valid)],
    verbose=500
)

print("\nAll 3 models trained!")

0:	learn: 0.0331884	test: 0.0205384	best: 0.0205384 (0)	total: 226ms	remaining: 26m 22s
500:	learn: 0.9371307	test: 0.7948579	best: 0.8045819 (222)	total: 38.8s	remaining: 8m 22s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.8045819229
bestIteration = 222

Shrink model to first 223 iterations.
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[193]	valid_0's l2: 0.00275002
[0]	validation_0-rmse:0.11137
[500]	validation_0-rmse:0.05280
[568]	validation_0-rmse:0.05301

All 3 models trained!


# Step 9 : Validation Score

In [18]:
# Individual model predictions (inverse log transform for scoring)
cat_preds = np.expm1(cat_model.predict(X_valid))
lgb_preds = np.expm1(lgb_model.predict(X_valid_lgb))
xgb_preds = np.expm1(xgb_model.predict(X_valid_lgb))

y_valid_orig = np.expm1(y_valid)

# Ensemble: weighted average
ensemble_preds = 0.5 * cat_preds + 0.25 * lgb_preds + 0.25 * xgb_preds
ensemble_preds = np.clip(ensemble_preds, 0, None)  # Clip negatives

print(f"CatBoost  R2: {r2_score(y_valid_orig, cat_preds):.6f}")
print(f"LightGBM  R2: {r2_score(y_valid_orig, lgb_preds):.6f}")
print(f"XGBoost   R2: {r2_score(y_valid_orig, xgb_preds):.6f}")
print(f"Ensemble  R2: {r2_score(y_valid_orig, ensemble_preds):.6f}")
print(f"\nCompetition Score: {r2_score(y_valid_orig, ensemble_preds) * 100:.4f}")

CatBoost  R2: 0.783618
LightGBM  R2: 0.755403
XGBoost   R2: 0.761114
Ensemble  R2: 0.774569

Competition Score: 77.4569


# Step 10 : Train on Full Dataset

In [19]:
# Retrain all 3 models on full data
final_cat = CatBoostRegressor(
    iterations=7000, learning_rate=0.02, depth=8,
    l2_leaf_reg=5, bagging_temperature=0.8, random_strength=1.0,
    min_data_in_leaf=20, loss_function='RMSE', random_seed=42, verbose=500
)
final_cat.fit(X, y, cat_features=cat_features)

final_lgb = LGBMRegressor(
    n_estimators=5000, learning_rate=0.02, max_depth=8, num_leaves=127,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=5, random_state=42, verbose=-1
)
final_lgb.fit(X_lgb, y, categorical_feature=lgb_cat_indices)

final_xgb = XGBRegressor(
    n_estimators=5000, learning_rate=0.02, max_depth=8,
    min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=5, random_state=42, verbosity=0,
    enable_categorical=True
)
final_xgb.fit(X_lgb, y)

print("All 3 final models trained on full data!")

0:	learn: 0.1070553	total: 85.5ms	remaining: 9m 58s
500:	learn: 0.0277895	total: 46.2s	remaining: 9m 59s
1000:	learn: 0.0258921	total: 1m 32s	remaining: 9m 13s
1500:	learn: 0.0246043	total: 2m 27s	remaining: 9m 1s
2000:	learn: 0.0237715	total: 3m 24s	remaining: 8m 30s
2500:	learn: 0.0231563	total: 4m 18s	remaining: 7m 44s
3000:	learn: 0.0226250	total: 5m 9s	remaining: 6m 52s
3500:	learn: 0.0221519	total: 6m 3s	remaining: 6m 2s
4000:	learn: 0.0217377	total: 6m 58s	remaining: 5m 13s
4500:	learn: 0.0213473	total: 7m 54s	remaining: 4m 23s
5000:	learn: 0.0210014	total: 8m 50s	remaining: 3m 31s
5500:	learn: 0.0206844	total: 9m 44s	remaining: 2m 39s
6000:	learn: 0.0204126	total: 10m 38s	remaining: 1m 46s
6500:	learn: 0.0201469	total: 11m 32s	remaining: 53.1s
6999:	learn: 0.0198757	total: 12m 22s	remaining: 0us
All 3 final models trained on full data!


# Step 11 : Generate Predictions

In [20]:
# Ensemble prediction with inverse log transform
cat_test_preds = np.expm1(final_cat.predict(X_test))
lgb_test_preds = np.expm1(final_lgb.predict(X_test_lgb))
xgb_test_preds = np.expm1(final_xgb.predict(X_test_lgb))

test_predictions = 0.5 * cat_test_preds + 0.25 * lgb_test_preds + 0.25 * xgb_test_preds
test_predictions = np.clip(test_predictions, 0, None)  # No negative demand

print(f"Predictions — min: {test_predictions.min():.4f}, max: {test_predictions.max():.4f}, mean: {test_predictions.mean():.4f}")

Predictions — min: 0.0014, max: 1.1100, mean: 0.1242


# Step 12 : Create Submission File

In [21]:
submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': test_predictions
})

submission.to_csv('submission.csv', index=False)

print(submission.head())

print("\nSubmission Shape:", submission.shape)

print("\nsubmission.csv generated successfully!")

   Index    demand
0      0  0.058675
1      1  0.024001
2      2  0.027248
3      3  0.036357
4      4  0.055860

Submission Shape: (41778, 2)

submission.csv generated successfully!


In [32]:
importance = final_model.get_feature_importance()

feature_importance = pd.DataFrame({
    'Feature': features,
    'Importance': importance
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nTop Features:\n")
print(feature_importance)


Top Features:

          Feature  Importance
2        RoadType   48.529810
0         geohash   13.903974
4   LargeVehicles    9.018574
3   NumberofLanes    6.207230
12      longitude    5.604919
11       latitude    4.546792
14       hour_cos    3.247089
13       hour_sin    3.056160
10  total_minutes    2.871804
1             day    1.034147
8            hour    1.026804
6     Temperature    0.353932
7         Weather    0.315950
9          minute    0.157367
5       Landmarks    0.125447
15     is_weekend    0.000000
